# 06 · Address generator (CPU)

**What this does:** writes the benchmark. Each target building gets one or more address strings of the kind people really type, and every string is stored with labels saying exactly what is in it.

**Files written**

| File | Rows | Used for |
|---|---|---|
| `train.parquet` | 2 addresses per training building | training models |
| `val.parquet` | 1 per validation building | tuning |
| `test.parquet` | four test sets in one file (column `split`) | all reported numbers |

**The four test sets**

| `split` | What | Why |
|---|---|---|
| `test_warm` | 3,000 unseen buildings in seen areas | everyday accuracy |
| `test_cold` | 3,000 buildings in areas never trained on | new-area accuracy |
| `test_rel` | a fixed number per landmark relation | rare relations (opposite, behind) get enough rows to measure |
| `test_real` | real addresses typed by OpenStreetMap mappers and (if notebook 02 ran) Overture business addresses | checks that results on generated text carry over to real text |

**Address styles:** formal (door number, street, locality, pincode), landmark ("opp X, near Y"), complex (flat in a named complex), offset ("150 m north of X"), minimal (what a hurried person types).

**Noise, each one labelled per row:** spelling slips, abbreviations, missing spaces, stray symbols, shuffled order, missing parts, Hindi or Telugu phrasing in Roman letters, shortened landmark names, wrong pincode (far or neighbouring), appended delivery instructions, and pure gibberish (correct response: no answer).

**What is invented and what is real:** places, streets and pincodes are always real map data. Flat numbers and door numbers are invented and flagged; rows with an invented door number are kept out of the comparison with commercial geocoders, because an invented number can collide with a real house elsewhere.

**Runtime:** about 5 minutes, CPU. **Needs:** notebooks 04 and 05.

In [ ]:
# Setup (the same cell opens every notebook): fetch the project code, install what is missing, find the data folder.
import os, sys, subprocess
REPO = 'SiddharthaRaja/Navigate'
PIP  = ['h3', 'rapidfuzz']
on_cloud = os.path.exists('/content') or os.path.exists('/kaggle/working')
if on_cloud:
    CODE = '/kaggle/working/Navigate-code' if os.path.exists('/kaggle/working') else '/content/Navigate'
    if not os.path.exists(f'{CODE}/bench'):
        token = os.environ.get('GH_TOKEN', '')
        try:                                   # only needed while the GitHub repo is private
            from google.colab import userdata; token = token or userdata.get('GH_TOKEN')
        except Exception: pass
        try:
            from kaggle_secrets import UserSecretsClient; token = token or UserSecretsClient().get_secret('GH_TOKEN')
        except Exception: pass
        url = f'https://{token}@github.com/{REPO}.git' if token else f'https://github.com/{REPO}.git'
        r = subprocess.run(['git', 'clone', '--depth', '1', url, CODE], env={**os.environ, 'GIT_TERMINAL_PROMPT': '0'})
        assert r.returncode == 0, 'Could not fetch the code. If the repo is private, add a GitHub token as the secret GH_TOKEN.'
    else:
        subprocess.run(['git', '-C', CODE, 'pull', '--ff-only'])
    if PIP: subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PIP], check=True)
else:
    CODE = os.path.abspath('..' if os.path.basename(os.getcwd()) == 'notebooks' else '.')
sys.path.insert(0, CODE)
import numpy as np, pandas as pd
from bench import env
P = env.setup('hyderabad')
print('running on', P['where'], '| data folder:', P['root'])

In [ ]:
from bench import textgen as tg
B, G = P['bench'], P['gaz']
t   = pd.read_parquet(f'{B}/targets.parquet')
rel = pd.read_parquet(f'{B}/relations.parquet')
lm  = pd.read_parquet(f'{B}/landmarks_clean.parquet')
import json
cold_cells = json.load(open(f'{B}/splits.json'))['cold_cells']
gen = tg.Generator(t, rel, lm, city='hyderabad', seed=7)

## 1. Training and validation text
`allow_cold=False` means: never mention a landmark that sits inside a cold cell.

In [ ]:
train = gen.generate(t[t.split == 'train'], 'train', n_variants=2, allow_cold=False, seed=1)
val   = gen.generate(t[t.split == 'val'],   'val',   n_variants=1, allow_cold=False, seed=2)
print(f"train {len(train):,} rows | val {len(val):,} rows")

## 2. Test sets

In [ ]:
N_TEST = 3000
hw, hc = t[t.split == 'holdout_warm'], t[t.split == 'holdout_cold']
test_warm = gen.generate(hw.sample(min(N_TEST, len(hw)), random_state=1), 'test_warm', seed=3, abstain_frac=0.02)
test_cold = gen.generate(hc.sample(min(N_TEST, len(hc)), random_state=1), 'test_cold', seed=4, abstain_frac=0.02)
left = pd.concat([hw[~hw.building_id.isin(test_warm.building_id)], hc[~hc.building_id.isin(test_cold.building_id)]])
test_rel  = tg.relation_test_set(gen, left, rel, per_relation=250)
test_real = tg.real_test_set(G, cold_cells, max_rows=3000)
test = pd.concat([test_warm, test_cold, test_rel, test_real], ignore_index=True)
test = tg.mark_api_subset(test)          # the stratified subset that is sent to paid geocoders
print(test.groupby('split').agg(rows=('id', 'size'), api_subset=('api_subset', 'sum')).to_string())

## 3. Checks that must pass

In [ ]:
cold_lm = set(lm[lm.in_cold_cell].landmark_id)
used = set(train.ref_landmark_id.dropna()) | set(train.get('ref2_landmark_id', pd.Series(dtype=str)).dropna())
assert not (used & cold_lm), 'training text mentions a landmark from a cold cell'
assert not (set(train.building_id) & set(test.building_id)), 'a building is in both train and test'
assert not (set(train.h3_7) & set(cold_cells)), 'a cold cell has training rows'
assert test.id.is_unique and train.id.is_unique
print('no leakage: OK')
print('\ntest_warm by style :', test_warm['style'].value_counts().to_dict())
print('test_warm by noise :', test_warm.noise_level.value_counts().to_dict())
print('test_rel by relation:', test_rel.relation.value_counts().to_dict())
print('landmark hidden from gazetteer (test_warm):', test_warm.ref_in_gazetteer.value_counts(dropna=False).to_dict())
print('\noracle distance = how far the best place named in the text is from the building (metres)')
print(test_warm.groupby('style').oracle_err_m.describe()[['count', '50%', '75%']].round(0).to_string())

## 4. Realism check (do this by eye, it is the gate for everything after)
Read the 40 addresses below next to real ones (your own delivery apps, shop websites). Ask of each: *would anyone type this?* If more than about 1 in 5 look machine-made, change the templates in `bench/textgen.py` before going further. Notebook 07 then rewrites a share of them with a language model to add variety.

In [ ]:
pd.set_option('display.max_colwidth', 140, 'display.width', 250)
show = test[test.split.isin(['test_warm', 'test_cold'])].sample(40, random_state=5)
for r in show.itertuples():
    print(f"[{r.style:8s} {r.noise_level:5s}] {r.text}")
print('\nreal addresses for comparison:')
for x in test[test.split == 'test_real'].text.sample(min(12, (test.split == 'test_real').sum()), random_state=1): print('  ', x)

## 5. Save

In [ ]:
train.to_parquet(f'{B}/train.parquet', index=False)
val.to_parquet(f'{B}/val.parquet', index=False)
test.to_parquet(f'{B}/test.parquet', index=False)
card = {'city': 'hyderabad', 'created': pd.Timestamp.now().strftime('%Y-%m-%d'), 'train_rows': len(train), 'val_rows': len(val),
        'test_rows': test.split.value_counts().to_dict(), 'api_subset_rows': int(test.api_subset.sum()),
        'sources': 'OpenStreetMap contributors (ODbL 1.0)' + ('; Overture Maps Places' if (test.source == 'overture_addr').any() else ''),
        'invented_tokens': 'flat numbers and door numbers (flagged per row in synthetic_tokens)'}
json.dump(card, open(f'{B}/dataset_card.json', 'w'), indent=2)
print(json.dumps(card, indent=2))

## 6. Optional: copy the data to Hugging Face
Only needed if you want to train on Kaggle (which cannot see your Google Drive). Create a token with write access at huggingface.co/settings/tokens, store it as the secret `HF_TOKEN`, set your repo name below and un-comment the line. The repo is created private.

In [ ]:
# env.push_to_hub(P, 'YOUR_HF_USERNAME/navigate-data')

## Share the outputs
Packs only the files this notebook created into one zip in `My Drive > Navigate > data > share`. Download that one file to share or back up the results.

In [ ]:
env.pack_outputs(P, '06' + '_outputs', [f"{P['bench']}/{f}" for f in ('train.parquet', 'val.parquet', 'test.parquet', 'dataset_card.json')])